In [149]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [150]:
def load_and_explore_dataset(file_path):
        print("-" * 60)
        print("LOAD AND EXPLORE DATE")
        print("-" * 60)

        df = pd.read_csv(file_path)
        print('shape of the dataset')
        print(df.shape)
        print('\ncheck missing')
        print(df.isnull().sum())
        print('\nFirst five row')
        print(df.head())
        print('\nDescriptive statistics')
        print(df.describe())
        print('\ndataset info')
        print(df.info())
        print('\nHouse Type distribution')
        print(df['house_type'].value_counts())
        print('\nFurnishing distribution')
        print(df['furnishing'].value_counts())

        return df

In [151]:
## Preprocessing
def preprocessing_data(df):
    print('\n' + "=" * 60)
    print("PREPROCESSING DATA")
    print("=" * 60)


    df_preprocessed = df.copy()
    df_preprocessed = df_preprocessed.dropna()


    label_encoder = {}
    df_columns = ["house_type", "region_parent_name"]
    for col in df_columns:
        le = LabelEncoder()
        df_preprocessed[col + "_encoded"] = le.fit_transform(df_preprocessed[col])
        label_encoder = le
        print(f"\n{col} Encoded")
        for i, label in enumerate(le.classes_):
            print(f"{label}: {i}")

    ordinal_encoder = OrdinalEncoder(
        categories=[
            ["Semi-Furnished", "Unfurnished", "Furnished"],
            ['Diamond', 'Enterprise', 'False', 'Vip', 'Vip_Gold', 'Premium', 'Basic']
        ]
    )

    df_preprocessed[["furnishing_encoded", "is_boost_encoded"]] = (
        ordinal_encoder.fit_transform(
            df_preprocessed[["furnishing", "is_boost"]]
        )
    )

    print("\nFurnishing Encoded")
    for i, label in enumerate(
        ordinal_encoder.categories_[0]
    ):
        print(f"{label}: {i}")

    print("\nBoost State Encoded")
    for i, label in enumerate(
        ordinal_encoder.categories_[0]
    ):
        print(f"{label}: {i}")
            
    print("Preprocessed data shape", df_preprocessed.shape)

    return df_preprocessed, label_encoder, ordinal_encoder

**Creating Data Label X and Y**

In [152]:
def featured_data(df_preprocessed):
    print('\n' + "=" * 60)
    print("FEATURED DATA")
    print("=" * 60)

    featured_columns = ["property_size(sqm)", "house_type_encoded", "bedrooms", "bathrooms", 
                        "furnishing_encoded", "region_parent_name", "is_boost_encoded"]
    target_columns = ["price"]

    x = df_preprocessed[featured_columns]
    y = df_preprocessed[target_columns]

    print("\nFeatures shape:", df_preprocessed[featured_columns].shape)
    print("\nTarget shape:", df_preprocessed[target_columns].shape)
    print("\nFeature columns:", featured_columns)

    return x, y, featured_columns

**Spliting the Data for Training and Testing**

In [153]:
def split_data(x, y, test_size=0.2, random_state=42):
    print('\n' + "=" * 60)
    print("SPLITING DATA")
    print("=" * 60)

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size:", x_train.shape[0])
    print("\nTest set size:", x_test.shape[0])

    print("\nTraining set price range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    print("\nTest set price range {:.2f} - {:.2f}".format(
        float(y_test.values.min()), float(y_test.values.max())
    ))

    return x_train, x_test, y_train, y_test

**Scaling the data to standardize numerical values**

In [154]:
#making use of standard scaler
def scale_data(x_train, x_test):
    print('\n' + "=" * 60)
    print("SCALE DATA")
    print("=" * 60)

    # Select numerical columns only
    numerical_cols = x_train.select_dtypes(include="number").columns
    
    scaler = StandardScaler()
    x_train_scaled = x_train.copy()
    x_test_scaled = x_test.copy()
    
    x_train_scaled[numerical_cols] = scaler.fit_transform(x_train[numerical_cols])
    x_test_scaled[numerical_cols] = scaler.transform(x_test[numerical_cols])

    x_train_scaled = pd.get_dummies(
        x_train_scaled,
        columns=["region_parent_name"],
        dtype=int
    )
    
    x_test_scaled = pd.get_dummies(
        x_test_scaled,
        columns=["region_parent_name"],
        dtype=int
    )

    print("\nFeature scaled successfully")
    print("\nTraining set scaled shape:", x_train_scaled.shape)
    print("\nTesting set scaled shape:", x_test_scaled.shape)

    return x_train_scaled, x_test_scaled, scaler

**Training the model - Linear Regression**

In [155]:
def train_model(x_train_scale, y_train, featured_columns):
    print('\n' + "=" * 60)
    print("TRAIN MODEL")
    print("=" * 60)

    model = LinearRegression()
    
    model.fit(x_train_scale, y_train)

    print("\nModel Trained Successfully")
    print("\nModel Coefficient")

    for feature, coef in zip(featured_columns, model.coef_.ravel()):
        print(f" {feature}: {coef:.2f}")
    print(f"\nModel Intercept: {float(model.intercept_[0]):.2f}")

    return model

In [156]:
def main():
    file_path = 'jiji_housing_cleaned.csv'

    ## load and explore datset
    df = load_and_explore_dataset(file_path)


    ##preprocessed
    df_preprocessed, label_encoder, ordinal_encoder = preprocessing_data(df)

    ##features data
    x, y, featured_columns = featured_data(df_preprocessed)

    ##split train and test data
    x_train, x_test, y_train, y_test = split_data(x, y)

    ##scaled 
    x_train_scaled, x_test_scaled, scaler = scale_data(x_train, x_test)

    ##training model
    model = train_model(x_train_scaled, y_train, featured_columns)

In [157]:
if __name__ == "__main__":
    main()


------------------------------------------------------------
LOAD AND EXPLORE DATE
------------------------------------------------------------
shape of the dataset
(1782, 11)

check missing
title                 0
property_size(sqm)    0
house_type            0
bedrooms              0
bathrooms             0
furnishing            0
region                0
region_name           0
region_parent_name    0
is_boost              0
price                 0
dtype: int64

First five row
                                               title  property_size(sqm)  \
0         2bdrm Maisonette in Lekki Phase 1 for sale                 185   
1             4bdrm Duplex in Thomas Estate for sale                 300   
2  5bdrm Duplex in Lekki Palm City, Ado / Ajah fo...                 300   
3  4bdrm Duplex in Golden Park Estate, Sangotedo ...                 300   
4                       5bdrm Duplex in Idu for sale                 700   

   house_type  bedrooms  bathrooms      furnishing         